In [9]:
# !pip install numpy pandas
# (numpy/pandas ship preinstalled in Colab; uncomment if running in a bare local environment)

# Step 6, Part A — Offline RL: Decision Transformer
### A1-A2: Building the synthetic trajectory dataset + return-to-go

**Where this fits:** this is the very first stage of Safe RL, per `SAFE_RL_GUIDE.md`. It does not
train anything yet — it only builds the dataset Decision Transformer will learn from in the next
session (Step A3). This notebook covers exactly Steps A1 and A2:

- **A1** — build a synthetic trajectory dataset around the `BL Baseline` anchor weights
- **A2** — attach return-to-go to every step, so each trajectory is in the `(return_to_go, state,
  action)` format Decision Transformer expects

**Anchor used, and why it changed:** `bl_baseline_train_weights.csv` — the plain Black-Litterman
baseline (equal-weight-implied prior, **no** World-Model or LLM-memory views), rolling-rebalanced
monthly over the **TRAIN** period (2018-01-03 -> 2024-12-31, 85 rebalance dates). This is a
deliberate change from an earlier pass of this stage, which anchored on `bl_llm_memory_weights.csv`
over the **TEST** period (2025-2026, 19 rebalance dates) — the same period this project's other
strategies are finally compared on. Pretraining and evaluating on the same period is a genuine
information leak, so the offline stage now trains only on TRAIN, leaving TEST completely untouched
until Part B's walk-forward evaluation (this deliberately departs from `SAFE_RL_GUIDE.md`'s literal
Step A1 instruction to use rebalance dates "across your TRAIN and TEST periods" — a project-level
methodology decision, not an oversight). The World-Model (Step 3) and LLM+memory (Step 5) view
pipelines were only ever run for the TEST-period rebalance dates, so they have no corresponding
output for TRAIN — the plain BL baseline is the correct (and only available) anchor for this stage.

Everything below is saved to disk at the end so the next session can load it directly and continue
with A3 (training the Decision Transformer) without rebuilding this part.

## Setup

In [10]:
import numpy as np
import pandas as pd
import pickle

np.random.seed(42)

ASSETS = ["EURUSD", "GBPUSD", "GOLD", "OIL"]
N_VARIANTS = 6           # synthetic trajectories per rebalance period, conservative -> aggressive
NOISE_SCALE = 0.035      # per-day, per-asset random nudge (see A1c for why this is smaller than
                         # an earlier pass used)
SKEW_SCALE = 0.18        # how strongly "aggressive" vs "conservative" tilts the ALLOCATION
                         # (not just the overall scale -- see A1c)
SKEW_DIRECTION = np.array([-1.0, -0.5, 0.5, 1.0])  # one entry per asset, aligned with ASSETS:
                         # aggressive variants tilt toward GOLD/OIL (higher historical vol),
                         # conservative variants tilt toward EURUSD/GBPUSD (lower historical vol)
                         # -- see A1c for why this replaced a uniform (same-sign) skew

df = pd.read_csv("project8_dataset_v2.csv", parse_dates=["Date"], index_col="Date")
prices = df[[f"{a}_Close" for a in ASSETS]].copy()
prices.columns = ASSETS
prices = prices.dropna()   # drop a day if ANY asset is missing it -- same convention as every
                            # other notebook in this project (GOLD/OIL have occasional NaNs from
                            # differing market holiday calendars; doing this AFTER pct_change
                            # instead of before would silently cut real trading days -- this order
                            # matches the rest of the project)
returns = prices.pct_change().dropna()
returns_clean = returns.clip(lower=-0.20, upper=0.20)   # same winsorization used everywhere else in this project

train_dates = df[df["PERIOD"] == "TRAIN"].index
train_rets = returns_clean.loc[returns_clean.index.isin(train_dates)]

anchor = pd.read_csv("bl_baseline_train_weights.csv", parse_dates=["Date"]).set_index("Date")
print("Anchor (rebalance-date) weights:", anchor.shape)
print("Daily TRAIN returns:", train_rets.shape)

Anchor (rebalance-date) weights: (85, 4)
Daily TRAIN returns: (1758, 4)


## A1a — Expand the anchor to daily granularity

`bl_baseline_train_weights.csv` has one row per *rebalance* date (85 monthly rebalances across
2018-2024) — but the portfolio actually *holds* those weights every day until the next rebalance,
exactly like the walk-forward backtests built earlier in this project. Decision Transformer needs
day-by-day trajectories to learn from, not 85 isolated points, so the first step is forward-filling
the rebalance-date weights into a full daily series (1,758 trading days) — this mirrors the real
backtest mechanics exactly (`current_w` stays fixed between rebalances) rather than inventing a new
assumption.

In [11]:
daily_anchor = anchor.reindex(train_rets.index, method="ffill")
# the very first TRAIN day should already have a rebalance (matches every other backtest in this
# project); guard against any leading NaN just in case the anchor's first date differs slightly
daily_anchor = daily_anchor.bfill()

assert daily_anchor.isna().sum().sum() == 0, "no day should be missing an anchor weight"
assert len(daily_anchor) == len(train_rets)
print(f"Expanded to {len(daily_anchor)} daily anchor rows")
daily_anchor.head(3)

Expanded to 1758 daily anchor rows


,EURUSD,GBPUSD,GOLD,OIL
Date,,,,
2018-01-03,0.25,0.25,0.25,0.25
2018-01-04,0.25,0.25,0.25,0.25
2018-01-05,0.25,0.25,0.25,0.25


## A1b — Per-day state features

State at day *t* uses only information available *before* that day's return is known — trailing
5-day mean return and volatility per asset (purely historical, no look-ahead), plus that day's
anchor weight (itself only ever set using data up to its own rebalance date, so also leak-free).
12 features total: 4 assets x (mean return, volatility) + 4 anchor weights.

In [12]:
def build_state_features(t_idx):
    '''State at day t_idx: trailing 5-day return stats (using days strictly before t_idx)
    plus that day's anchor weight. No information from day t_idx's own return is used.'''
    if t_idx < 5:
        trailing = train_rets.iloc[:t_idx] if t_idx > 0 else train_rets.iloc[:1] * 0
    else:
        trailing = train_rets.iloc[t_idx - 5:t_idx]
    mean_ret = trailing[ASSETS].mean().values if len(trailing) > 0 else np.zeros(len(ASSETS))
    vol = trailing[ASSETS].std().fillna(0).values if len(trailing) > 0 else np.zeros(len(ASSETS))
    w = daily_anchor.iloc[t_idx][ASSETS].values
    return np.concatenate([mean_ret, vol, w]).astype(np.float32)

state_dim = build_state_features(10).shape[0]
print(f"State dimension: {state_dim} (4 mean returns + 4 vols + 4 anchor weights)")
print("Example state at t=10:", build_state_features(10).round(4))

State dimension: 12 (4 mean returns + 4 vols + 4 anchor weights)
Example state at t=10: [0.005  0.0034 0.004  0.0032 0.0086 0.0089 0.0034 0.0072 0.25   0.25
 0.25   0.25  ]


## A1c — Build the synthetic trajectories

`N_VARIANTS` synthetic "investors" per day, spanning conservative -> aggressive relative to the
anchor -- this is what gives Decision Transformer a genuine *spread* of achieved returns to learn
from later (Step A4 checks that this spread is real, not just noise). Each variant is a full
1,758-day trajectory: `(state, action, reward)` at every day.

**Why `SKEW_DIRECTION` exists (a fix made during this TRAIN-range pass):** an earlier version of
this cell applied the *same* skew to every asset (`delta = skew * scale`, identical for all 4
assets). That washes out almost entirely once the action is renormalized to sum to 1 — pushing
every asset up by the same amount and then dividing by a sum close to 1 barely changes the
*relative* weights, so "aggressive" and "conservative" variants ended up looking nearly identical
after renormalization, in expectation (verified directly: with a uniform skew, the conservative and
most-aggressive variants' own mean EURUSD weight over the full TRAIN period differed by under
0.004). `SAFE_RL_GUIDE.md`'s own troubleshooting section names this exact failure mode:
*"if every variant is a tiny nudge around the same BL anchor, Decision Transformer never sees a
real spread of outcomes... make the aggressive/conservative variants genuinely different, not just
noisy."* `SKEW_DIRECTION` fixes this by tilting the *allocation* itself: aggressive variants shift
weight toward GOLD/OIL (the two higher-historical-volatility assets in this universe), conservative
variants shift it toward EURUSD/GBPUSD — a real, renormalization-surviving difference. `NOISE_SCALE`
was also reduced (0.08 -> 0.035) so this deliberate signal isn't swamped by per-day randomness.
Step A4 (next notebook) confirms this actually produces a model that uses return-to-go.

In [13]:
def build_synthetic_trajectories(daily_anchor, train_rets, n_variants=N_VARIANTS, noise_scale=NOISE_SCALE,
                                  skew_scale=SKEW_SCALE, skew_direction=SKEW_DIRECTION, seed=42):
    rng = np.random.default_rng(seed)
    T = len(train_rets)
    trajectories = []

    for variant in range(n_variants):
        skew = (variant / (n_variants - 1)) - 0.5   # -0.5 (conservative) .. +0.5 (aggressive)
        traj = []
        for t in range(T):
            anchor_w = daily_anchor.iloc[t][ASSETS].values
            delta = rng.normal(loc=skew * skew_scale * skew_direction, scale=noise_scale, size=len(ASSETS))
            action = np.clip(anchor_w + delta, 0, 1)
            action = action / action.sum()   # renormalize -- stay long-only, sum-to-1

            reward = float(action @ train_rets.iloc[t][ASSETS].values)
            state = build_state_features(t)

            traj.append({"t": t, "date": train_rets.index[t], "state": state,
                         "action": action.astype(np.float32), "reward": reward})
        trajectories.append(traj)
        total_ret = np.prod([1 + s["reward"] for s in traj]) - 1
        mean_w = np.mean([s["action"] for s in traj], axis=0)
        print(f"  variant {variant} (skew={skew:+.2f}): total period return = {total_ret:+.2%}  "
              f"| mean weights {dict(zip(ASSETS, mean_w.round(3)))}")

    return trajectories

trajectories = build_synthetic_trajectories(daily_anchor, train_rets)
print(f"\nBuilt {len(trajectories)} trajectories x {len(trajectories[0])} days each")

  variant 0 (skew=-0.50): total period return = +24.38%  | mean weights {'EURUSD': np.float32(0.353), 'GBPUSD': np.float32(0.296), 'GOLD': np.float32(0.203), 'OIL': np.float32(0.148)}
  variant 1 (skew=-0.30): total period return = +28.02%  | mean weights {'EURUSD': np.float32(0.315), 'GBPUSD': np.float32(0.279), 'GOLD': np.float32(0.221), 'OIL': np.float32(0.184)}
  variant 2 (skew=-0.10): total period return = +46.44%  | mean weights {'EURUSD': np.float32(0.279), 'GBPUSD': np.float32(0.26), 'GOLD': np.float32(0.239), 'OIL': np.float32(0.222)}
  variant 3 (skew=+0.10): total period return = +33.82%  | mean weights {'EURUSD': np.float32(0.244), 'GBPUSD': np.float32(0.242), 'GOLD': np.float32(0.257), 'OIL': np.float32(0.257)}
  variant 4 (skew=+0.30): total period return = +44.18%  | mean weights {'EURUSD': np.float32(0.208), 'GBPUSD': np.float32(0.223), 'GOLD': np.float32(0.274), 'OIL': np.float32(0.295)}
  variant 5 (skew=+0.50): total period return = +62.02%  | mean weights {'EURUSD'

### Sanity check: do the variants actually span a real spread of outcomes?

If every variant produced nearly the same total return, Decision Transformer would have nothing
to learn from — return-to-go conditioning only works if different actions genuinely led to
different results in the training data.

In [14]:
total_returns = [np.prod([1 + s["reward"] for s in traj]) - 1 for traj in trajectories]
spread = max(total_returns) - min(total_returns)
print(f"Total-return spread across variants: {spread:.2%} (min {min(total_returns):+.2%}, max {max(total_returns):+.2%})")
assert spread > 0.02, "variants are too similar -- widen NOISE_SCALE/SKEW_SCALE or the skew range"
print("PASS: variants span a real, usable spread of outcomes")

# constraint check across every single trajectory/day
all_actions = np.array([s["action"] for traj in trajectories for s in traj])
print(f"\nAction constraint check across {len(all_actions)} (trajectory x day) actions:")
print(f"  sum-to-1 max deviation: {np.abs(all_actions.sum(axis=1) - 1).max():.2e}")
print(f"  min action value: {all_actions.min():.6f} (should be >= 0)")
print(f"  any NaN: {np.isnan(all_actions).any()}")

Total-return spread across variants: 37.64% (min +24.38%, max +62.02%)
PASS: variants span a real, usable spread of outcomes

Action constraint check across 10548 (trajectory x day) actions:
  sum-to-1 max deviation: 1.19e-07
  min action value: 0.000000 (should be >= 0)
  any NaN: False


## A2 — Attach return-to-go

For every step in every trajectory, `return_to_go` is the sum of that trajectory's *remaining*
rewards from this day to the end — the target Decision Transformer will later learn to condition
its action on ("given you want to end up with this much return from here, what do you do today").

In [15]:
def add_return_to_go(trajectories):
    for traj in trajectories:
        rewards = [step["reward"] for step in traj]
        rtg = np.cumsum(rewards[::-1])[::-1]
        for step, r in zip(traj, rtg):
            step["return_to_go"] = float(r)
    return trajectories

trajectories = add_return_to_go(trajectories)

# sanity checks
t0 = trajectories[0]
assert abs(t0[0]["return_to_go"] - sum(s["reward"] for s in t0)) < 1e-9, "day-0 RTG must equal the trajectory's total reward"
assert abs(t0[-1]["return_to_go"] - t0[-1]["reward"]) < 1e-9, "last day's RTG must equal just that day's reward"
rtgs = [s["return_to_go"] for s in t0]
print("First 5 return-to-go values (should generally trend down as the trajectory progresses):")
print([round(r, 4) for r in rtgs[:5]])
print("Last 5:", [round(r, 4) for r in rtgs[-5:]])
print("\nPASS: return-to-go boundary conditions correct")

First 5 return-to-go values (should generally trend down as the trajectory progresses):
[0.249, 0.2416, 0.2431, 0.2422, 0.2422]
Last 5: [0.0055, 0.005, 0.0049, 0.0035, 0.0022]

PASS: return-to-go boundary conditions correct


## Save the trajectory dataset

Saved as-is so the next session can load it directly and continue straight into Step A3 (training
Decision Transformer) without rebuilding A1-A2.

In [16]:
with open("offline_trajectories.pkl", "wb") as f:
    pickle.dump({
        "trajectories": trajectories,
        "assets": ASSETS,
        "state_dim": state_dim,
        "n_variants": N_VARIANTS,
        "noise_scale": NOISE_SCALE,
        "skew_scale": SKEW_SCALE,
        "skew_direction": SKEW_DIRECTION,
        "dates": list(train_rets.index),
    }, f)

print(f"Saved offline_trajectories.pkl: {len(trajectories)} trajectories x {len(trajectories[0])} days, "
      f"state_dim={state_dim}")
print("\nReady for Step A3 (Decision Transformer training) in the next session.")

Saved offline_trajectories.pkl: 6 trajectories x 1758 days, state_dim=12

Ready for Step A3 (Decision Transformer training) in the next session.
